# Homography Pipeline — Simplified (production)

Same fixes, same results, as the full debugging notebook -- code cleaned of anything not actually
used (no dead methods, no unused variables), split into one cell per step:

- `top_left_corner` keypoint (was missing)
- Ambiguous line pairings resolved via a pose-based anchor (only the empirically reliable pose
  indices), falling back to the previous frame's H, falling back to unresolved dual-candidate
- Confidence/jump-adaptive EMA smoothing, seeded only from anchor-resolved frames
- Self-calibrating orientation invariant (no manual reference frame) that catches any remaining
  left-right mirror flips


## 0. Setup

In [ ]:
!pip install -q ultralytics==8.3.203 opencv-python-headless tqdm

In [ ]:
import os

SEG_MODEL_PATH = "/kaggle/input/datasets/zeinsaad/homography/seg_model.pt"
POSE_MODEL_PATH = "/kaggle/input/datasets/zeinsaad/homography/pose_model.pt"
VIDEO_PATH = "/kaggle/input/datasets/zeinsaad/homography/clip.mkv"
HOMOGRAPHY_CACHE_PATH = "/kaggle/working/homography_cache.pkl"

os.makedirs("homography", exist_ok=True)
for p in (SEG_MODEL_PATH, POSE_MODEL_PATH, VIDEO_PATH):
    print(("\u2705 " if os.path.exists(p) else "\u274c MISSING: ") + p)


## 1. `homography/keypoints.py`

In [ ]:
%%writefile homography/keypoints.py
from __future__ import annotations


def build_pitch_keypoints(pitch_length: float, pitch_width: float) -> dict[str, tuple[float, float]]:
    return {
        "top_left_corner":               (0.0, 0.0),
        "top_right_corner":              (pitch_length, 0.0),
        "halfway_top":                   (pitch_length / 2, 0.0),
        "halfway_bottom":                (pitch_length / 2, pitch_width),
        "center_spot":                   (pitch_length / 2, pitch_width / 2),
        "left_big_rect_top_outer":       (0.0, 13.85),
        "left_big_rect_top_inner":       (16.5, 13.85),
        "left_big_rect_bottom_outer":    (0.0, 54.15),
        "left_big_rect_bottom_inner":    (16.5, 54.15),
        "right_big_rect_top_outer":      (pitch_length, 13.85),
        "right_big_rect_top_inner":      (pitch_length - 16.5, 13.85),
        "right_big_rect_bottom_outer":   (pitch_length, 54.15),
        "right_big_rect_bottom_inner":   (pitch_length - 16.5, 54.15),
        "right_pen_spot":                (pitch_length - 11.0, 34.0),
        "left_small_rect_top_outer":     (0.0, 24.85),
        "left_small_rect_top_inner":     (5.5, 24.85),
        "left_small_rect_bottom_outer":  (0.0, 43.15),
        "left_small_rect_bottom_inner":  (5.5, 43.15),
        "right_small_rect_top_outer":    (pitch_length, 24.85),
        "right_small_rect_top_inner":    (pitch_length - 5.5, 24.85),
        "right_small_rect_bottom_outer": (pitch_length, 43.15),
        "right_small_rect_bottom_inner": (pitch_length - 5.5, 43.15),
        "left_big_rect_main_upper":      (16.5, 24.85),
        "left_big_rect_main_lower":      (16.5, 43.15),
        "right_big_rect_main_upper":     (pitch_length - 16.5, 24.85),
        "right_big_rect_main_lower":     (pitch_length - 16.5, 43.15),
        "circle_top":                    (pitch_length / 2, pitch_width / 2 - 9.15),
        "circle_bottom":                 (pitch_length / 2, pitch_width / 2 + 9.15),
        "circle_left":                   (pitch_length / 2 - 9.15, pitch_width / 2),
        "circle_right":                  (pitch_length / 2 + 9.15, pitch_width / 2),
    }


LINE_PAIR_TO_KEYPOINT = {
    "Side line top":          {"type": "endpoints", "keys": ["top_left_corner", "top_right_corner"]},
    "Middle line":            {"type": "endpoints", "keys": ["halfway_top", "halfway_bottom"]},
    "Circle central":         {"type": "centroid",  "keys": ["center_spot"]},
    "Big rect. left top":     {"type": "endpoints", "keys": ["left_big_rect_top_outer",      "left_big_rect_top_inner"]},
    "Big rect. left bottom":  {"type": "endpoints", "keys": ["left_big_rect_bottom_outer",   "left_big_rect_bottom_inner"]},
    "Big rect. left main":    {"type": "endpoints", "keys": ["left_big_rect_top_inner",      "left_big_rect_bottom_inner"]},
    "Big rect. right top":    {"type": "endpoints", "keys": ["right_big_rect_top_outer",     "right_big_rect_top_inner"]},
    "Big rect. right bottom": {"type": "endpoints", "keys": ["right_big_rect_bottom_outer",  "right_big_rect_bottom_inner"]},
    "Big rect. right main":   {"type": "endpoints", "keys": ["right_big_rect_top_inner",     "right_big_rect_bottom_inner"]},
    "Small rect. left top":   {"type": "endpoints", "keys": ["left_small_rect_top_outer",    "left_small_rect_top_inner"]},
    "Small rect. left bottom":{"type": "endpoints", "keys": ["left_small_rect_bottom_outer", "left_small_rect_bottom_inner"]},
    "Small rect. left main":  {"type": "endpoints", "keys": ["left_small_rect_top_inner",    "left_small_rect_bottom_inner"]},
    "Small rect. right top":  {"type": "endpoints", "keys": ["right_small_rect_top_outer",   "right_small_rect_top_inner"]},
    "Small rect. right bottom":{"type":"endpoints", "keys": ["right_small_rect_bottom_outer","right_small_rect_bottom_inner"]},
    "Small rect. right main": {"type": "endpoints", "keys": ["right_small_rect_top_inner",   "right_small_rect_bottom_inner"]},
}


def build_pose_keypoints(pitch_keypoints_real: dict[str, tuple[float, float]]) -> dict[int, tuple[float, float]]:
    return {
        9:  pitch_keypoints_real["left_big_rect_top_inner"],
        12: pitch_keypoints_real["left_big_rect_bottom_inner"],
        13: pitch_keypoints_real["halfway_top"],
        14: pitch_keypoints_real["circle_top"],
        15: pitch_keypoints_real["circle_bottom"],
        16: pitch_keypoints_real["halfway_bottom"],
        17: pitch_keypoints_real["right_big_rect_top_inner"],
        20: pitch_keypoints_real["right_big_rect_bottom_inner"],
        21: pitch_keypoints_real["right_pen_spot"],
        30: pitch_keypoints_real["circle_left"],
        31: pitch_keypoints_real["circle_right"],
    }


# Reliable subset used to build the anchor that resolves left/right line-pairing
# ambiguity -- right-side box/penalty-spot/circle points (17,20,21,30,31) were
# empirically less trustworthy than these.
TRUSTED_ANCHOR_POSE_INDICES = {9, 12, 13, 14, 15, 16}


## 2. `homography/config.py`

In [ ]:
%%writefile homography/config.py
from dataclasses import dataclass


@dataclass
class HomographyConfig:
    seg_model_path: str = ""
    pose_model_path: str = ""
    video_path: str = ""
    output_cache_path: str = ""

    conf_thresh_seg: float = 0.25
    conf_thresh_pose: float = 0.20
    img_size: int = 960

    px_per_meter: int = 10
    ransac_thresh: float = 25.0
    pitch_length: float = 105.0
    pitch_width: float = 68.0

    ema_alpha: float = 0.3
    min_inliers_full_confidence: int = 15
    min_alpha: float = 0.05
    max_alpha: float = 0.9
    jump_px_threshold: float = 25.0
    jump_confidence_threshold: float = 0.5
    jump_alpha: float = 0.8

    min_anchor_points: int = 4
    min_anchor_inlier_ratio: float = 0.7


## 3. `homography/engine.py`

In [ ]:
%%writefile homography/engine.py
from __future__ import annotations

import cv2
import numpy as np
from ultralytics import YOLO

from .config import HomographyConfig
from .keypoints import (
    LINE_PAIR_TO_KEYPOINT,
    TRUSTED_ANCHOR_POSE_INDICES,
    build_pitch_keypoints,
    build_pose_keypoints,
)


class HomographyEngine:
    def __init__(self, config: HomographyConfig):
        self.config = config
        self.seg_model: YOLO | None = None
        self.pose_model: YOLO | None = None
        self.pitch_keypoints_real = build_pitch_keypoints(config.pitch_length, config.pitch_width)
        self.pose_keypoints_real = build_pose_keypoints(self.pitch_keypoints_real)
        self.reference_orientation_sign: float | None = None

    def load_models(self) -> None:
        cfg = self.config
        self.seg_model = YOLO(cfg.seg_model_path)
        self.pose_model = YOLO(cfg.pose_model_path)

    # -- correspondences -------------------------------------------------

    def get_correspondences(self, frame: np.ndarray, bootstrap_H: np.ndarray | None = None):
        """Extract (image_pts, world_pts_m, labels). Ambiguous line pairings
        are resolved via: (1) a pose-based anchor built only from the
        reliable subset, validated by inlier ratio; (2) bootstrap_H (the
        previous frame's H) if no anchor; (3) unresolved dual-candidate as
        last resort."""
        cfg = self.config
        sr = self.seg_model.predict(frame, conf=cfg.conf_thresh_seg, imgsz=cfg.img_size, verbose=False)[0]
        pr = self.pose_model.predict(frame, conf=cfg.conf_thresh_pose, imgsz=cfg.img_size, verbose=False)[0]

        pose_i, pose_w, pose_l, pose_idx = self._extract_pose(pr)
        centroid_i, centroid_w, centroid_l = self._extract_seg_centroids(sr)
        endpoint_lines = self._extract_seg_endpoint_candidates(sr)

        trusted = np.array([i in TRUSTED_ANCHOR_POSE_INDICES for i in pose_idx], dtype=bool)
        anchor_i = self._vstack(pose_i[trusted] if len(pose_i) else pose_i, centroid_i)
        anchor_w = self._vstack(pose_w[trusted] if len(pose_w) else pose_w, centroid_w)

        H_resolver, tag = None, "unresolved"
        if len(anchor_i) >= cfg.min_anchor_points:
            H_cand, mask_cand = self._compute_homography(anchor_i, anchor_w)
            if H_cand is not None:
                ratio = float(mask_cand.sum()) / len(mask_cand)
                if ratio >= cfg.min_anchor_inlier_ratio and int(mask_cand.sum()) >= cfg.min_anchor_points:
                    H_resolver, tag = H_cand, "anchor"
        if H_resolver is None and bootstrap_H is not None:
            H_resolver, tag = bootstrap_H, "bootstrap"

        resolved_i, resolved_w, resolved_l = [], [], []
        for p1, p2, key0, key1 in endpoint_lines:
            w0 = np.array(self.pitch_keypoints_real[key0], np.float32)
            w1 = np.array(self.pitch_keypoints_real[key1], np.float32)
            if H_resolver is not None:
                proj = cv2.perspectiveTransform(
                    np.array([[p1], [p2]], np.float32), H_resolver
                ).reshape(2, 2) / cfg.px_per_meter
                if np.linalg.norm(proj[0]-w1)+np.linalg.norm(proj[1]-w0) < np.linalg.norm(proj[0]-w0)+np.linalg.norm(proj[1]-w1):
                    p1, p2 = p2, p1
                resolved_i += [p1, p2]; resolved_w += [w0, w1]
                resolved_l += [f"{key0}_{tag}", f"{key1}_{tag}"]
            else:
                resolved_i += [p1, p2, p1, p2]; resolved_w += [w0, w1, w1, w0]
                resolved_l += [f"{key0}_unresolved"] * 2 + [f"{key1}_unresolved"] * 2

        resolved_i = np.array(resolved_i, np.float32) if resolved_i else np.empty((0, 2), np.float32)
        resolved_w = np.array(resolved_w, np.float32) if resolved_w else np.empty((0, 2), np.float32)

        ai = self._vstack(self._vstack(pose_i, centroid_i), resolved_i)
        aw = self._vstack(self._vstack(pose_w, centroid_w), resolved_w)
        return ai, aw, pose_l + centroid_l + resolved_l

    @staticmethod
    def _vstack(a, b):
        if len(a) and len(b): return np.vstack([a, b])
        return a if len(a) else b

    def _compute_homography(self, ai, aw):
        if len(ai) < 4: return None, None
        cfg = self.config
        return cv2.findHomography(ai, aw * cfg.px_per_meter, cv2.RANSAC, ransacReprojThreshold=cfg.ransac_thresh)

    # -- public API --------------------------------------------------------

    def get_homography(self, frame: np.ndarray, bootstrap_H: np.ndarray | None = None) -> np.ndarray | None:
        ai, aw, _ = self.get_correspondences(frame, bootstrap_H)
        H, _ = self._compute_homography(ai, aw)
        return self._enforce_reference_orientation(H)

    def get_homography_debug(self, frame: np.ndarray, bootstrap_H: np.ndarray | None = None):
        ai, aw, labels = self.get_correspondences(frame, bootstrap_H)
        H, mask = self._compute_homography(ai, aw)
        return self._enforce_reference_orientation(H), mask, ai, aw, labels

    def pixel_to_pitch(self, H, px, py):
        pt = cv2.perspectiveTransform(np.array([[[px, py]]], np.float32), H).reshape(2)
        return float(pt[0] / self.config.px_per_meter), float(pt[1] / self.config.px_per_meter)

    # -- orientation invariant, self-calibrating ----------------------------

    def _orientation_sign(self, H):
        pts = np.array([[[200.0, 540.0]], [[1700.0, 540.0]]], np.float32)
        proj = cv2.perspectiveTransform(pts, H).reshape(2, 2)
        return float(np.sign(proj[1, 0] - proj[0, 0]))

    def calibrate_reference_orientation_auto(self, video_path, sample_stride=50, max_samples=60, min_votes=5):
        """No manual reference frame needed: samples frames across the whole
        clip, votes (weighted by inlier count) on the majority orientation,
        and locks that in as the reference every H is checked against."""
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
        idxs = list(range(0, total, max(sample_stride, 1)))[:max_samples]

        prev, self.reference_orientation_sign = self.reference_orientation_sign, None
        votes, n_valid = {1.0: 0.0, -1.0: 0.0}, 0
        try:
            for idx in idxs:
                cap = cv2.VideoCapture(video_path)
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read(); cap.release()
                if not ret: continue
                H, mask, _, _, _ = self.get_homography_debug(frame)
                if H is None: continue
                sign = self._orientation_sign(H)
                if sign == 0: continue
                votes[sign] += float(mask.sum()) if mask is not None else 1.0
                n_valid += 1
        finally:
            self.reference_orientation_sign = prev

        if n_valid < min_votes:
            raise RuntimeError(f"Only {n_valid} valid samples -- can't calibrate reliably.")

        winner = 1.0 if votes[1.0] >= votes[-1.0] else -1.0
        total_w = votes[1.0] + votes[-1.0]
        agreement = 100 * votes[winner] / total_w if total_w else 0.0
        self.reference_orientation_sign = winner
        print(f"Orientation calibrated from {n_valid} frames -> sign={winner:+.0f} ({agreement:.1f}% agreement)")
        if agreement < 70.0:
            print("\u26A0\uFE0F  Low agreement -- pipeline may be unreliable on this clip.")
        return winner

    def _enforce_reference_orientation(self, H):
        if H is None or self.reference_orientation_sign is None:
            return H
        sign = self._orientation_sign(H)
        if sign == 0 or sign == self.reference_orientation_sign:
            return H
        L_px = self.config.pitch_length * self.config.px_per_meter
        F = np.array([[-1.0, 0.0, L_px], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float64)
        return (F @ H).astype(H.dtype)

    # -- internal extraction -----------------------------------------------

    def _extract_seg_centroids(self, results):
        if results.masks is None:
            return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32), []
        IP, WP, LB = [], [], []
        names = self.seg_model.names
        for mxy, ci in zip(results.masks.xy, results.boxes.cls):
            spec = LINE_PAIR_TO_KEYPOINT.get(names[int(ci)])
            if spec is None or spec["type"] != "centroid": continue
            key = spec["keys"][0]
            if key in self.pitch_keypoints_real:
                IP.append(self._centroid(mxy)); WP.append(self.pitch_keypoints_real[key]); LB.append(key)
        return np.array(IP, np.float32), np.array(WP, np.float32), LB

    def _extract_seg_endpoint_candidates(self, results):
        if results.masks is None: return []
        out = []
        names = self.seg_model.names
        for mxy, ci in zip(results.masks.xy, results.boxes.cls):
            spec = LINE_PAIR_TO_KEYPOINT.get(names[int(ci)])
            if spec is None or spec["type"] != "endpoints" or len(mxy) < 2: continue
            key0, key1 = spec["keys"]
            if key0 not in self.pitch_keypoints_real or key1 not in self.pitch_keypoints_real: continue
            p1, p2 = self._pca_endpoints(mxy)
            out.append((p1, p2, key0, key1))
        return out

    def _extract_pose(self, results):
        if results.keypoints is None or len(results.keypoints) == 0:
            return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32), [], []
        IP, WP, LB, IDX = [], [], [], []
        kpts = results.keypoints.xy[0].cpu().numpy()
        confs = results.keypoints.conf[0].cpu().numpy() if results.keypoints.conf is not None else np.ones(len(kpts))
        for idx, ((x, y), c) in enumerate(zip(kpts, confs)):
            if c < self.config.conf_thresh_pose or (x == 0 and y == 0) or idx not in self.pose_keypoints_real:
                continue
            IP.append([x, y]); WP.append(self.pose_keypoints_real[idx]); LB.append(f"pose_{idx}"); IDX.append(idx)
        return np.array(IP, np.float32), np.array(WP, np.float32), LB, IDX

    @staticmethod
    def _centroid(mask_xy):
        return mask_xy.astype(np.float32).mean(axis=0)

    @staticmethod
    def _pca_endpoints(mask_xy):
        p = mask_xy.astype(np.float32)
        c = p - p.mean(axis=0)
        _, _, vt = np.linalg.svd(c, full_matrices=False)
        proj = c @ vt[0]
        return p[np.argmin(proj)], p[np.argmax(proj)]


## 4. `homography/cache_io.py`

In [ ]:
%%writefile homography/cache_io.py
from __future__ import annotations

import os, pickle
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm

from .engine import HomographyEngine


def save_cache(cache, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f: pickle.dump(cache, f)
    print(f"\U0001F4BE Saved to '{path}'.")


def load_cache(path):
    with open(path, "rb") as f: return pickle.load(f)


def _disagreement_px(H_a, H_b, pitch_keypoints_real, px_per_meter):
    pts = (np.array(list(pitch_keypoints_real.values()), np.float32) * px_per_meter).reshape(-1, 1, 2)
    try:
        pa = cv2.perspectiveTransform(pts, np.linalg.inv(H_a)).reshape(-1, 2)
        pb = cv2.perspectiveTransform(pts, np.linalg.inv(H_b)).reshape(-1, 2)
    except np.linalg.LinAlgError:
        return float("inf")
    return float(np.median(np.linalg.norm(pa - pb, axis=1)))


def _tag(labels):
    if any(l.endswith("_unresolved") for l in labels): return "unresolved"
    if any(l.endswith("_bootstrap") for l in labels): return "bootstrap"
    if any(l.endswith("_anchor") for l in labels): return "anchor"
    return "none"


def build_cache(engine: HomographyEngine, video_path: str, ema_alpha: float) -> list:
    """Confidence/jump-adaptive EMA, seeded only from anchor-resolved frames.
    Assumes engine.calibrate_reference_orientation_auto() was already called."""
    if engine.seg_model is None:
        engine.load_models()

    cfg = engine.config
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    smoothed, H_ema = [None] * total, None
    n_jumps, valid = 0, 0

    for i in tqdm(range(total), desc="Homography"):
        ret, frame = cap.read()
        if not ret:
            smoothed[i] = H_ema.copy() if H_ema is not None else None
            continue

        H_raw, mask, _, _, labels = engine.get_homography_debug(frame, bootstrap_H=H_ema)
        tag = _tag(labels)

        if H_raw is not None:
            valid += 1
            n_in = int(mask.sum()) if mask is not None else 0
            conf = min(1.0, n_in / max(cfg.min_inliers_full_confidence, 1))
            if H_ema is None:
                if tag == "anchor":
                    H_ema = H_raw.copy()
            else:
                jump = _disagreement_px(H_raw, H_ema, engine.pitch_keypoints_real, cfg.px_per_meter) > cfg.jump_px_threshold \
                       and conf >= cfg.jump_confidence_threshold
                alpha = ema_alpha * conf
                if jump:
                    alpha = max(alpha, cfg.jump_alpha); n_jumps += 1
                alpha = float(np.clip(alpha, cfg.min_alpha, cfg.max_alpha))
                H_ema = alpha * H_raw + (1 - alpha) * H_ema

        smoothed[i] = H_ema.copy() if H_ema is not None else None

    cap.release()
    n_ok = sum(1 for h in smoothed if h is not None)
    print(f"Valid: {valid}/{total} | Final coverage: {n_ok}/{total} ({100*n_ok/total:.1f}%) | Jumps corrected: {n_jumps}")
    return smoothed


def get_or_build_cache(engine, video_path, cache_path, ema_alpha=0.3, force_rebuild=False):
    if os.path.exists(cache_path) and not force_rebuild:
        print(f"\u2705 Loaded cache from '{cache_path}'.")
        return load_cache(cache_path)
    cache = build_cache(engine, video_path, ema_alpha)
    save_cache(cache, cache_path)
    return cache


In [ ]:
%%writefile homography/__init__.py
from .config import HomographyConfig
from .engine import HomographyEngine
from .cache_io import build_cache, get_or_build_cache, save_cache, load_cache


## 5. Imports

In [ ]:
import sys, cv2, numpy as np
import matplotlib.pyplot as plt

for mod in list(sys.modules):
    if mod == "homography" or mod.startswith("homography."):
        del sys.modules[mod]  # avoid stale module cache across kernel re-runs
sys.path.insert(0, "/kaggle/working")

from homography import HomographyConfig, HomographyEngine, get_or_build_cache


## 6. Instantiate engine and load models

In [ ]:
config = HomographyConfig(
    seg_model_path=SEG_MODEL_PATH, pose_model_path=POSE_MODEL_PATH,
    video_path=VIDEO_PATH, output_cache_path=HOMOGRAPHY_CACHE_PATH,
)
engine = HomographyEngine(config)
engine.load_models()


## 7. Calibrate orientation (self-calibrating, no manual reference frame)

In [ ]:
engine.calibrate_reference_orientation_auto(VIDEO_PATH, sample_stride=50, max_samples=60)


## 8. Build the full cache

In [ ]:
full_cache = get_or_build_cache(
    engine, VIDEO_PATH, HOMOGRAPHY_CACHE_PATH,
    ema_alpha=config.ema_alpha, force_rebuild=True,
)


## 9. Quick visual sanity check

In [ ]:
def grab_frame(video_path, idx):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read(); cap.release()
    return frame

def draw_overlay(frame, H, cfg, color=(0, 255, 0)):
    vis = frame.copy()
    if H is None:
        cv2.putText(vis, "NO H", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        return vis
    L, W, r = cfg.pitch_length, cfg.pitch_width, 9.15
    segs = [
        [(0,0),(L,0),(L,W),(0,W),(0,0)], [(L/2,0),(L/2,W)],
        [(0,13.85),(16.5,13.85),(16.5,54.15),(0,54.15)],
        [(L,13.85),(L-16.5,13.85),(L-16.5,54.15),(L,54.15)],
        [(0,24.85),(5.5,24.85),(5.5,43.15),(0,43.15)],
        [(L,24.85),(L-5.5,24.85),(L-5.5,43.15),(L,43.15)],
        [(L/2+r*np.cos(t), W/2+r*np.sin(t)) for t in np.linspace(0, 2*np.pi, 40)],
    ]
    H_inv = np.linalg.inv(H)
    for seg in segs:
        pts_px = cv2.perspectiveTransform((np.array(seg, np.float32)*cfg.px_per_meter).reshape(-1,1,2), H_inv).reshape(-1,2)
        cv2.polylines(vis, [pts_px.astype(np.int32)], False, color, 2)
    return vis

total = len(full_cache)
idxs = [5, total//4, total//2, 3*total//4, total-10]
fig, axes = plt.subplots(1, len(idxs), figsize=(4*len(idxs), 4))
for ax, idx in zip(axes, idxs):
    frame = grab_frame(VIDEO_PATH, idx)
    vis = draw_overlay(frame, full_cache[idx], config)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.set_title(f"frame {idx}"); ax.axis("off")
plt.tight_layout(); plt.show()
